# Установка зависимостей

In [1]:
%pip install \
  --index-url https://download.pytorch.org/whl/cu126 \
  --extra-index-url https://pypi.org/simple \
  torch==2.9.1 \
  torchvision==0.24.1 \
  torchaudio==2.9.1 \
  "transformers>=4.51,<5.0" \
  "huggingface_hub>=0.30,<1.0" \
  "Pillow>=11.0,<12.0" \
  "tqdm>=4.67,<5.0" \
  "jupyterlab>=4.4,<5.0" \
  "ipykernel>=6.29,<7.0" \
  "numpy>=2.2,<3.0" \
  "manuscript-ocr[dev]==0.1.10"

Looking in indexes: https://download.pytorch.org/whl/cu126, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 186.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 505.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 474.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 488.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 364.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 520.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 396.9 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 506.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 629.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━

In [5]:
pip uninstall manuscript-ocr -y

Found existing installation: manuscript-ocr 0.1.10
Uninstalling manuscript-ocr-0.1.10:
  Successfully uninstalled manuscript-ocr-0.1.10
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install "manuscript-ocr[dev] @ git+https://github.com/konstantinkozhin/manuscript-ocr.git@v_0_1_11"

  Cloning https://github.com/konstantinkozhin/manuscript-ocr.git (to revision v_0_1_11) to /tmp/pip-install-lxwkxm7o/manuscript-ocr_52a71f31ddad4bef92ce923697cab698
  Running command git clone --filter=blob:none --quiet https://github.com/konstantinkozhin/manuscript-ocr.git /tmp/pip-install-lxwkxm7o/manuscript-ocr_52a71f31ddad4bef92ce923697cab698
  Running command git checkout -b v_0_1_11 --track origin/v_0_1_11
  Переключились на новую ветку «v_0_1_11»
  branch 'v_0_1_11' set up to track 'origin/v_0_1_11'.
  Resolved https://github.com/konstantinkozhin/manuscript-ocr.git to commit 49b8308b9743b43f5866416782c5d66eba17f710
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


# Детекция номерных знаков

### Скачивание и распаковка набора данных

In [2]:
from huggingface_hub import snapshot_download
import zipfile
import os

DATA_DIR = "data/car_plate_dataset"

# создать папку если её нет
os.makedirs(DATA_DIR, exist_ok=True)

# скачать датасет
snapshot_download(
    repo_id="AY000554/Car_plate_detecting_dataset",
    repo_type="dataset",
    local_dir=DATA_DIR,
    local_dir_use_symlinks=False
)

# распаковать архивы
for file in ["train.zip", "val.zip", "test.zip"]:
    zip_path = os.path.join(DATA_DIR, file)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

print("Dataset downloaded and extracted to /data")

/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 7 files: 100%|██████████| 7/7 [1:42:20<00:00, 877.21s/it] 


Dataset downloaded and extracted to /data


### Уточнение полигонов с помощью SAM

In [1]:
from pathlib import Path
import json
import random

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import SamProcessor, SamModel

DATASET_ROOT = Path("data/car_plate_dataset")
OUT_ROOT = DATASET_ROOT / "coco_sam_v2"
ANN_DIR = OUT_ROOT / "annotations"
SAMPLE_DIR = OUT_ROOT / "samples"

SPLITS = ["train", "val", "test"]
SAMPLES_PER_SPLIT = 10
SEED = 42
IMAGE_EXTS = {".jpg", ".jpeg", ".bmp", ".png"}

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
model = SamModel.from_pretrained("facebook/sam-vit-base").to(device)
model.eval()


def order_points(pts):
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    return rect


def yolo_to_bbox(line, w, h):
    cls, x, y, bw, bh = map(float, line.split())

    x_center = x * w
    y_center = y * h
    bw *= w
    bh *= h

    x1 = max(0, int(x_center - bw / 2))
    y1 = max(0, int(y_center - bh / 2))
    x2 = min(w, int(x_center + bw / 2))
    y2 = min(h, int(y_center + bh / 2))

    bbox_xyxy = [x1, y1, x2, y2]
    bbox_xywh = [x1, y1, x2 - x1, y2 - y1]
    return int(cls), bbox_xyxy, bbox_xywh


def bbox_to_polygon4(bbox_xywh):
    x, y, w, h = bbox_xywh
    pts = np.array([
        [x, y],
        [x + w, y],
        [x + w, y + h],
        [x, y + h]
    ], dtype=np.float32)
    return order_points(pts)


def sam_mask(image, bbox_xyxy):
    inputs_cpu = processor(image, input_boxes=[[bbox_xyxy]], return_tensors="pt")
    inputs = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs_cpu.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)

    masks = processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs_cpu["original_sizes"],
        inputs_cpu["reshaped_input_sizes"]
    )

    mask_candidates = masks[0][0]
    if isinstance(mask_candidates, torch.Tensor):
        mask_candidates = mask_candidates.cpu().numpy()

    scores = outputs.iou_scores[0, 0].detach().cpu().numpy()

    if mask_candidates.ndim == 2:
        return mask_candidates > 0

    best_idx = int(np.argmax(scores))
    return mask_candidates[best_idx] > 0


def mask_to_polygon4(mask, bbox_xywh):
    x, y, w, h = map(int, bbox_xywh)

    pad = max(2, int(0.03 * max(w, h)))
    x0 = max(0, x - pad)
    y0 = max(0, y - pad)
    x1 = min(mask.shape[1], x + w + pad)
    y1 = min(mask.shape[0], y + h + pad)

    roi = (mask[y0:y1, x0:x1].astype(np.uint8)) * 255
    if roi.size == 0 or roi.max() == 0:
        polygon = bbox_to_polygon4(bbox_xywh)
        area = float(w * h)
        return [polygon.flatten().round(2).tolist()], area

    roi = cv2.morphologyEx(
        roi,
        cv2.MORPH_CLOSE,
        np.ones((3, 3), np.uint8),
        iterations=1
    )

    contours, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        polygon = bbox_to_polygon4(bbox_xywh)
        area = float(w * h)
        return [polygon.flatten().round(2).tolist()], area

    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) < 10:
        polygon = bbox_to_polygon4(bbox_xywh)
        area = float(w * h)
        return [polygon.flatten().round(2).tolist()], area

    hull = cv2.convexHull(contour)
    peri = cv2.arcLength(hull, True)

    best_quad = None
    best_area = -1.0

    for ratio in np.linspace(0.005, 0.08, 40):
        approx = cv2.approxPolyDP(hull, ratio * peri, True).reshape(-1, 2)

        if len(approx) == 4:
            approx = approx.astype(np.float32)
            quad_area = cv2.contourArea(approx.reshape(-1, 1, 2))
            if quad_area > best_area:
                best_quad = approx
                best_area = quad_area

    if best_quad is None:
        rect = cv2.minAreaRect(hull)
        best_quad = cv2.boxPoints(rect)

    best_quad[:, 0] += x0
    best_quad[:, 1] += y0
    best_quad = order_points(best_quad)

    area = float(mask.sum())
    return [best_quad.flatten().round(2).tolist()], area


def save_sample(image_path, anns, save_path, alpha=0.35):
    img = cv2.imread(str(image_path))
    overlay = img.copy()

    for ann in anns:
        overlay[ann["mask"]] = (0, 0, 255)

    vis = cv2.addWeighted(img, 1 - alpha, overlay, alpha, 0)

    for ann in anns:
        x, y, w, h = map(int, ann["bbox"])
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

        pts = np.array(ann["segmentation"][0], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(vis, [pts], True, (0, 255, 255), 2)

        for p in pts.reshape(-1, 2):
            cv2.circle(vis, tuple(p), 4, (0, 255, 255), -1)

    cv2.imwrite(str(save_path), vis)


def collect_classes():
    cls_ids = set()

    for split in SPLITS:
        for txt_path in (DATASET_ROOT / split / "labels").glob("*.txt"):
            for line in txt_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    cls_ids.add(int(float(line.split()[0])))

    cls_ids = sorted(cls_ids) if cls_ids else [0]
    cls2cat = {cls_id: i + 1 for i, cls_id in enumerate(cls_ids)}

    categories = [
        {
            "id": cls2cat[cls_id],
            "name": "license_plate" if cls_id == 0 else f"class_{cls_id}",
            "supercategory": "vehicle"
        }
        for cls_id in cls_ids
    ]

    return cls2cat, categories


def convert_split(split, cls2cat, categories):
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    images = []
    annotations = []
    vis_anns_by_image = {}
    image_records = []
    ann_id = 1

    image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

    for image_id, image_path in enumerate(
        tqdm(image_paths, desc=f"{split} images", unit="img", dynamic_ncols=True),
        start=1
    ):
        image = Image.open(image_path).convert("RGB")
        img_np = np.array(image)
        h, w = img_np.shape[:2]

        images.append({
            "id": image_id,
            "file_name": image_path.name,
            "width": w,
            "height": h
        })
        image_records.append((image_id, image_path))

        label_path = label_dir / f"{image_path.stem}.txt"
        vis_anns = []

        if label_path.exists():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue

                cls_id, bbox_xyxy, bbox_xywh = yolo_to_bbox(line, w, h)
                mask = sam_mask(image, bbox_xyxy)
                segmentation, area = mask_to_polygon4(mask, bbox_xywh)

                ann = {
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": cls2cat[cls_id],
                    "segmentation": segmentation,
                    "area": round(area, 2),
                    "bbox": [round(v, 2) for v in bbox_xywh],
                    "iscrowd": 0
                }

                annotations.append(ann)
                vis_anns.append({
                    "bbox": ann["bbox"],
                    "segmentation": segmentation,
                    "mask": mask
                })
                ann_id += 1

        vis_anns_by_image[image_id] = vis_anns

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    ANN_DIR.mkdir(parents=True, exist_ok=True)
    json_path = ANN_DIR / f"{split}.json"
    json_path.write_text(json.dumps(coco, ensure_ascii=False, indent=2), encoding="utf-8")

    split_sample_dir = SAMPLE_DIR / split
    split_sample_dir.mkdir(parents=True, exist_ok=True)

    candidates = [x for x in image_records if vis_anns_by_image[x[0]]]
    rng = random.Random(f"{SEED}-{split}")
    chosen = rng.sample(candidates, min(SAMPLES_PER_SPLIT, len(candidates)))

    for image_id, image_path in chosen:
        save_sample(image_path, vis_anns_by_image[image_id], split_sample_dir / image_path.name)

    print(f"{split}: {len(images)} images, {len(annotations)} annotations")


cls2cat, categories = collect_classes()

for split in SPLITS:
    convert_split(split, cls2cat, categories)

print("done")
print("device:", device)
print("json:", ANN_DIR.resolve())
print("samples:", SAMPLE_DIR.resolve())


/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
train images: 100%|██████████| 20505/20505 [36:53<00:00,  9.26img/s]


train: 20505 images, 21442 annotations


val images: 100%|██████████| 2563/2563 [04:37<00:00,  9.22img/s]


val: 2563 images, 2678 annotations


test images: 100%|██████████| 2564/2564 [04:37<00:00,  9.23img/s]


test: 2564 images, 2685 annotations
done
device: cuda
json: /home/user/snap/manuscript-ocr-license-plates/data/car_plate_dataset/coco_sam_v2/annotations
samples: /home/user/snap/manuscript-ocr-license-plates/data/car_plate_dataset/coco_sam_v2/samples


## Обучение модели

In [ ]:
from manuscript.detectors import EAST

train_images = r"data/car_plate_dataset/train/images"
train_annotations = r"data/car_plate_dataset/coco_sam_v2/annotations/train.json"
val_images = r"data/car_plate_dataset/val/images"
val_annotations = r"data/car_plate_dataset/coco_sam_v2/annotations/val.json"

EAST.train(
    train_images=train_images,
    train_anns=train_annotations,
    val_images=val_images,
    val_anns=val_annotations,
    model_name="EAST_LICENSE_PLATES_V1",
    epochs=90,
    batch_size=4,
    lr_scheduler="cosine",
    resume_from="/home/user/snap/manuscript-ocr-license-plates/experiments/EAST_LICENSE_PLATES_V1",
    #resume_from="/home/user/snap/manuscript-ocr-license-plates/east_50_g1.pth",
    early_stop=25,
    augmentation_config={
        "vflip_prob":            0.1,
        "mosaic_prob":           0.01, 
        "mosaic_center_range":   (0.3, 0.7),
        "cutmix_prob":           0.05,
        "cutmix_alpha":          1.0,
        "resizemix_prob":        0.01,
        "resizemix_scale_range": (0.15, 0.6),
    },
)

/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


Training configuration saved to: /home/user/snap/manuscript-ocr-license-plates/experiments/EAST_LICENSE_PLATES_V1/training_config.json


/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/manuscript/detectors/_east/train_utils.py:371: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Pre-training validation: Loss=6.0815, Dice=0.9069


Train 83:   0%|          | 0/5127 [00:00<?, ?it/s]/home/user/snap/manuscript-ocr-license-plates/env/lib/python3.12/site-packages/manuscript/detectors/_east/train_utils.py:374: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast()
Train 90: 100%|██████████| 5127/5127 [28:14<00:00,  3.03it/s]


Attempting to export best model to ONNX...
Loading checkpoint from /home/user/snap/manuscript-ocr-license-plates/experiments/EAST_LICENSE_PLATES_V1/checkpoints/best_loss.pth...
Auto-detecting backbone architecture...
   Detected 114 layer3 parameters
   Auto-detected backbone: resnet50

Loading PyTorch model...
Loaded pretrained model from /home/user/snap/manuscript-ocr-license-plates/experiments/EAST_LICENSE_PLATES_V1/checkpoints/best_loss.pth
Model architecture: EASTWrapper
Input shape: torch.Size([1, 3, 1024, 1024])
Output shapes:
  - score_map: torch.Size([1, 1, 256, 256])
  - geo_map: torch.Size([1, 8, 256, 256])

Exporting to ONNX (opset 14)...
Failed to export ONNX model: No module named 'onnxscript'


EASTModel(
  (backbone): ResNetFeatureExtractor(
    (extractor): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Module(
        (0): Module(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_ru